# SmolVLA on Colab: parcel task, dataset A1

Fine-tunes `lerobot/smolvla_base` on the private dataset
`Low101/parcel-sort-a` (100 episodes, 3 cameras: top, wrist, side).
One task text: "sort the parcel so the label faces down and push it
off the table".

Checkpoints go to Google Drive, so you can resume after a disconnect.

Before you start:
1. Upload the dataset from the laptop as a private repo (see the
   laptop command in the chat / plan notes).
2. Select Runtime > Change runtime type > GPU (A100 or L4 is best).

In [ ]:
!nvidia-smi
!nproc
!python --version   # LeRobot needs Python >= 3.12

## 1. Install LeRobot (same commit as the laptop)

In [ ]:
!git clone -q https://github.com/huggingface/lerobot.git /content/lerobot
%cd /content/lerobot
!git checkout -q 64b23178d5348609c266250d3e1f511eba4c33ff
!apt-get -qq install -y ffmpeg
!pip install -q -e ".[smolvla,training,dataset]"
# Same as the working ACT notebook (2026-10-21): these break the import.
!pip uninstall -y -q torchao diffusers

## 2. Hugging Face login
Select "Log in with your browser" (this worked in the ACT notebook).
The token must read the private dataset. Write access is necessary for
the model upload in step 7.

In [ ]:
!hf auth login

In [ ]:
from huggingface_hub import HfApi
info = HfApi().dataset_info("Low101/parcel-sort-a")
print("dataset found, private =", info.private)

## 3. Google Drive for checkpoints
Allow the access in the Google pop-up. Without the mount,
`/content/drive/MyDrive` is only a normal folder on the Colab disk,
and a disconnect deletes all checkpoints (the ACT run of 2026-10-21
was lost like this). The cell stops if Drive is not mounted.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Drive is NOT mounted"
DRIVE = "/content/drive/MyDrive/lerobot_runs"
os.makedirs(DRIVE, exist_ok=True)
with open(f"{DRIVE}/_write_test.txt", "w") as f:
    f.write("ok")
print("Drive mounted and writable:", DRIVE)
!df -h /content/drive | tail -1

## 4. Settings
- `STEPS = 20000`. Checkpoint every 5000 steps, or every 2500 steps
  on a small GPU (T4), where a disconnect costs more time.
- On Drive, each checkpoint keeps the model (869 MB). Only the newest
  also keeps the optimizer state (394 MB), which resume needs. A full
  run on a T4 uses about 7.4 GB of Drive.
- The batch size comes from the GPU memory. SmolVLA uses about
  0.3 GB per sample (measured on the laptop: batch 12 = 4.7 GB).
- Free Colab (T4, 2 CPUs): batch 16. The 2 CPUs decode the AV1 video
  of 3 cameras, so a bigger batch is not faster per sample. Expect
  about 2-3 s per step, so 20k steps need 2 or 3 sessions.
- Free Colab stops a session after about 12 h, or earlier if the tab
  is idle. Keep the tab open. After a stop, use step 6.
- The cameras map to the pretrained camera inputs: top = camera1,
  wrist = camera2, side = camera3. Use the same map at inference.

In [ ]:
import os
import torch

RUN = "smolvla_parcel_a1_20k"
STEPS = 20000
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH = 64 if gpu_gb >= 38 else 48 if gpu_gb >= 22 else 16
SAVE_FREQ = 5000 if gpu_gb >= 22 else 2500
WORKERS = max(2, os.cpu_count() - 1)
# Drive cannot store the "last" symlink, so training writes to the
# local disk and a thread copies each finished checkpoint to Drive.
OUT_DIR = f"/content/runs/{RUN}"
DRIVE_DIR = f"{DRIVE}/{RUN}"
RENAME = ('{"observation.images.top": "observation.images.camera1", '
          '"observation.images.wrist": "observation.images.camera2", '
          '"observation.images.side": "observation.images.camera3"}')
print(RUN, OUT_DIR, "->", DRIVE_DIR)
print(f"GPU {gpu_gb:.0f} GB -> batch {BATCH}, workers {WORKERS}")

In [ ]:
# Checkpoint copy to Drive (runs in the background during training).
import shutil
import threading
import time
from pathlib import Path

LOCAL_CK = Path(OUT_DIR) / "checkpoints"
DRIVE_CK = Path(DRIVE_DIR) / "checkpoints"


def finished_steps():
    last = LOCAL_CK / "last"
    if not last.is_symlink():
        return []
    newest = int(last.resolve().name)
    return sorted(p for p in LOCAL_CK.iterdir()
                  if p.name.isdigit() and int(p.name) <= newest)


def sync_once():
    DRIVE_CK.mkdir(parents=True, exist_ok=True)
    for src in finished_steps():
        dst = DRIVE_CK / src.name
        if dst.exists():
            continue
        tmp = DRIVE_CK / (src.name + ".part")
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.copytree(src, tmp)
        tmp.rename(dst)
        print(f"[drive] copied checkpoint {src.name}", flush=True)
        # Resume needs only the newest optimizer state (394 MB each).
        for old in DRIVE_CK.iterdir():
            if old.name.isdigit() and old.name < src.name:
                shutil.rmtree(old / "training_state", ignore_errors=True)


def sync_loop(stop):
    while not stop.is_set():
        try:
            sync_once()
        except Exception as e:  # keep training if Drive has a problem
            print("[drive] copy failed:", e, flush=True)
        stop.wait(60)


def start_sync():
    stop = threading.Event()
    threading.Thread(target=sync_loop, args=(stop,), daemon=True).start()
    return stop

## 5. Train (first start only)
Do not run this cell again after a disconnect. Use step 6 instead.
Each checkpoint is copied to Drive within about 1 minute after the
save. The last cell line copies the final checkpoint.

In the log, look at `data_s`. If `data_s` is near `updt_s` or higher,
the CPU video decode is the limit (AV1, 3 cameras). Then a smaller
batch gives the same speed per sample.

In [ ]:
assert os.path.ismount("/content/drive"), "run step 3 first"
stop = start_sync()
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=Low101/parcel-sort-a \
  --rename_map='{RENAME}' \
  --policy.device=cuda \
  --policy.push_to_hub=true \
  --policy.repo_id=Low101/{RUN} \
  --policy.private=true \
  --output_dir={OUT_DIR} \
  --job_name={RUN} \
  --steps={STEPS} \
  --save_freq={SAVE_FREQ} \
  --log_freq=100 \
  --batch_size={BATCH} \
  --num_workers={WORKERS} \
  --wandb.enable=false
stop.set()
sync_once()

## 6. Resume after a disconnect
1. Run steps 1 to 4 again (install, login, Drive, settings and the
   checkpoint copy cell).
2. Run this cell. It copies the newest checkpoint from Drive to the
   local disk, then continues the training from it.

In [ ]:
assert os.path.ismount("/content/drive"), "run step 3 first"
saved = sorted(p for p in DRIVE_CK.iterdir() if p.name.isdigit())
assert saved, "no checkpoint on Drive yet: start again with step 5"
newest = saved[-1]
local = LOCAL_CK / newest.name
if not local.exists():
    shutil.copytree(newest, local)
last = LOCAL_CK / "last"
if last.is_symlink():
    last.unlink()
last.symlink_to(newest.name)
print("resume from step", newest.name)

stop = start_sync()
!lerobot-train \
  --config_path={OUT_DIR}/checkpoints/last/pretrained_model/train_config.json \
  --resume=true
stop.set()
sync_once()

## 7. Upload all checkpoints (private)
The final model is pushed at the end of training. This cell also
uploads each checkpoint `pretrained_model` to `checkpoints/<step>/` in
the same private repo. The optimizer states stay on Drive.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

api = HfApi()
repo_id = f"Low101/{RUN}"
api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
api.update_repo_settings(repo_id, private=True)

ckpt_root = DRIVE_CK
steps = sorted(p.name for p in ckpt_root.iterdir() if p.name.isdigit())
for step in steps:
    api.upload_folder(
        repo_id=repo_id,
        folder_path=str(ckpt_root / step / "pretrained_model"),
        path_in_repo=f"checkpoints/{step}",
        commit_message=f"checkpoint {step}",
    )
    print("uploaded", step)
print("private:", api.model_info(repo_id).private)

## 8. Get the model on the laptop
```bash
cd ~/lerobot/lerobot
hf download Low101/smolvla_parcel_a1_20k \
  --local-dir outputs/train/smolvla_parcel_a1_20k/pretrained_model
```
One checkpoint, for example 10000:
```bash
hf download Low101/smolvla_parcel_a1_20k \
  --include "checkpoints/010000/*" \
  --local-dir outputs/train/smolvla_parcel_a1_20k
```
The same files are on Google Drive in
`lerobot_runs/smolvla_parcel_a1_20k/checkpoints/<step>/`.